<a href="https://colab.research.google.com/github/Rahullogic/handwritten-equation-solver/blob/main/Handwritten_equation_solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/handwritten-equation-solver-data'

Mounted at /content/drive


In [2]:
!git config --global user.email "rahulkumar0909007@gmail.com"
!git config --global user.name "Rahul"

In [3]:
from google.colab import userdata
TOKEN = userdata.get('GITHUB_TOKEN')

!git clone https://{TOKEN}@github.com/Rahullogic/handwritten-equation-solver.git
%cd handwritten-equation-solver

Cloning into 'handwritten-equation-solver'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 0), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.
/content/handwritten-equation-solver


In [4]:
!pwd

/content/handwritten-equation-solver


In [5]:
%cd /content/handwritten-equation-solver
!pwd   # should print /content/handwritten-equation-solver

/content/handwritten-equation-solver
/content/handwritten-equation-solver


In [39]:
%%writefile .gitignore
data/
checkpoints/
*.pt
*.pth
__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store

Overwriting .gitignore


In [40]:
!cat .gitignore

data/
checkpoints/
*.pt
*.pth
__pycache__/
.ipynb_checkpoints/
*.pyc
.DS_Store


In [22]:
!git add .gitignore
!git commit -m "Add .gitignore"
!git push

[main (root-commit) 204f5e9] Add .gitignore
 1 file changed, 8 insertions(+)
 create mode 100644 .gitignore
Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 2 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (3/3), 278 bytes | 278.00 KiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Rahullogic/handwritten-equation-solver.git
 * [new branch]      main -> main


In [23]:
!mkdir -p checkpoints
!touch checkpoints/dummy_test.pt
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [24]:
!rm checkpoints/dummy_test.pt

In [8]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [9]:
!kaggle datasets download -d xainano/handwrittenmathsymbols

Dataset URL: https://www.kaggle.com/datasets/xainano/handwrittenmathsymbols
License(s): CC0-1.0
100% 343M/343M [00:03<00:00, 109MB/s]



In [38]:
!ls /content/handwritten-equation-solver

checkpoints  handwrittenmathsymbols.zip


In [10]:
import shutil
shutil.move('/content/handwritten-equation-solver/handwrittenmathsymbols.zip', f'{DATA_DIR}/handwrittenmathsymbols.zip')

'/content/drive/MyDrive/handwritten-equation-solver-data/handwrittenmathsymbols.zip'

In [11]:
import zipfile
with zipfile.ZipFile(f'{DATA_DIR}/handwrittenmathsymbols.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/symbols_data')

In [12]:
!ls {DATA_DIR}

handwrittenmathsymbols.zip  symbols


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [15]:
torch.manual_seed(42)

In [18]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28,28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5),(0.5))
])

In [19]:
data_dir= '/content/symbols_data/extracted_images'

In [20]:
dataset= datasets.ImageFolder(root= data_dir, transform= transform)

In [22]:
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)


print(f"Successfully loaded {len(dataset)} images.")
print(f"The model will learn to recognize these classes: {dataset.classes}")

Successfully loaded 96429 images.
The model will learn to recognize these classes: ['!', '(', ')', '+', ',', '-', '0']


In [25]:
class MyNN(nn.Module):

  def __init__(self, input_features, num_classes):

    super().__init__()

    self.features= nn.Sequential(
        nn.Conv2d(input_features, 32, kernel_size=3, padding= 'same'),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),

        nn.Conv2d(32, 64, kernel_size=3, padding='same'),
        nn.ReLU(),
        nn.BatchNorm2d(64),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

    self.classifier= nn.Sequential(
        nn.Flatten(),
        nn.Linear(64*7*7, 128),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(128,64),
        nn.ReLU(),
        nn.Dropout(p=0.4),

        nn.Linear(64,num_classes)
    )

  def forward(self, X):

    X= self.features(X)
    X= self.classifier(X)

    return X

In [31]:
learning_rate=0.01
epochs= 10

In [30]:
model= MyNN(1,len(dataset.classes))
model.to(device)

criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [34]:
for epoch in range(epochs):

  model.train()
  running_loss = 0.0
  correct_predictions = 0
  total_samples = 0

  for images,labels in dataloader:

    images= images.to(device)
    labels= labels.to(device)

    outputs= model(images)

    loss= criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    running_loss += loss.item() * images.size(0)

    _, predicted = torch.max(outputs, 1)
    total_samples += labels.size(0)
    correct_predictions += (predicted == labels).sum().item()

  epoch_loss = running_loss / total_samples
  epoch_acc = (correct_predictions / total_samples) * 100

  print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

print("Training Completed");





Epoch [1/10] | Loss: 0.1873 | Accuracy: 95.36%
Epoch [2/10] | Loss: 0.1877 | Accuracy: 95.60%
Epoch [3/10] | Loss: 0.1924 | Accuracy: 95.84%
Epoch [4/10] | Loss: 0.1587 | Accuracy: 96.21%
Epoch [5/10] | Loss: 0.1583 | Accuracy: 96.39%
Epoch [6/10] | Loss: 0.1444 | Accuracy: 96.69%
Epoch [7/10] | Loss: 0.1527 | Accuracy: 96.61%
Epoch [8/10] | Loss: 0.1561 | Accuracy: 96.62%
Epoch [9/10] | Loss: 0.1537 | Accuracy: 96.58%
Epoch [10/10] | Loss: 0.1560 | Accuracy: 96.61%
Training Completed


In [35]:
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (4): ReLU()
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=7, bias=True)
  )
)

In [37]:

save_path = '/content/drive/MyDrive/Handwritten_equation_solver/symbol_classifier.pth'

torch.save(model.state_dict(), save_path)
print("Model saved successfully!")

Model saved successfully!
